# Week 02 Lab 04: Deep Research  
One of the classic cross-business Agentic use cases!  
A Deep Research Agent is broadly applicable to any business area, and to your own day-to-day activities. You can make use of this yourself!

In [1]:
from agents import Agent, WebSearchTool, trace, Runner, gen_trace_id, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from typing import Dict
from IPython.display import display, Markdown

In [2]:
load_dotenv(override=True)

True

## OpenAI Hosted Tools  
OpenAI SDK includes the following hosted tools:  
- The **Web Search** tool `web_search` lets an agent search the web.  
- The **File Search Retrieval** tool `file_search` allows retrieving information from OpenAI Vector Stores.  
- The **Computer use** tool `computer_use` allows automating computer use tasks like taking screenshots and clicking.  
- The **Code Interpreter** tool `code_interpreter` helps execute code in a sandbox for data anlysis, math, plotting, file processing, etc.  
- The **Image Generation** tool `image_generation` allows to Generate and edit images using OpenAI's image models.  
- The **Remote MCP** tool `mcp` enables the connection to external tools and services via the Model Context Protocol (MCP)  

### Important Note - API charge of WebSearchTool  
This is costing (at the time of development) 2.5 cents per call for OpenAI WebSearchTool. That can add up to $2 - $3 for the next 2 labs. We'll use free and low cost Search tools with other platforms, so feel free to skip running this if the cost is a concern.  

Costs are here: https://platform.openai.com/docs/pricing#web-search

In [3]:
instructions = "You are a research assistant. Given a search term, you search the web for that term and \
    produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 \
    words. Capture the main points. Write succintly, no need to have complete sentences or good \
    grammer. This will be consumed by someone sythesizing a report, so it's vital you capture the \
    seenence and ignore any fluff. Do not include any additional commentary other than summary itself."

search_agent = Agent(
    name= 'Search Agent',
    instructions = instructions,
    tools = [WebSearchTool(search_context_size = 'low')],
    model = 'gpt-4o-mini',
    model_settings = ModelSettings(tool_choice= 'required')
)

In [4]:
message = "Latest AI Agent frameworks in 2025"
with trace("Search"):
    result = await Runner.run(search_agent, message)

display(Markdown(result.final_output))

In 2025, several AI agent frameworks have emerged, each offering unique features for developing autonomous AI systems. LangChain, with a 30% market share, is recognized for its modular architecture and extensive ecosystem, facilitating the creation of LLM-powered applications and autonomous agents. LangGraph, a component of LangChain, introduces graph-based workflows for complex, stateful, and branching tasks. ([artificial-intelligence-wiki.com](https://artificial-intelligence-wiki.com/ai-automation/ai-workflow-orchestration-tools/ai-agent-frameworks-guide/?utm_source=openai))

CrewAI, holding a 20% market share, focuses on role-based multi-agent collaboration, enabling the definition of roles, goals, and communication protocols among agents, making it suitable for research assistants and task automation setups. AutoGPT, with a 25% market share, emphasizes autonomous goal-driven agents capable of self-improvement and adaptation. AutoGen, developed by Microsoft, is designed for conversational multi-agent orchestration, supporting human-in-the-loop and Azure-native workflows. ([artificial-intelligence-wiki.com](https://artificial-intelligence-wiki.com/ai-automation/ai-workflow-orchestration-tools/ai-agent-frameworks-guide/?utm_source=openai))

The OpenAI Agents SDK offers a managed runtime with first-party tools, facilitating rapid prototyping on the OpenAI stack. LlamaIndex Agents are built around retrieval-augmented generation (RAG), ideal for knowledge-intensive tasks requiring accurate data grounding. LightAgent is a lightweight, open-source framework integrating core functionalities like memory, tools, and Tree of Thought, enabling seamless integration with mainstream chat platforms. ([trendix.tech](https://www.trendix.tech/ai-agent-framework/?utm_source=openai))

In addition to these frameworks, major tech companies have introduced platforms to enhance AI agent development. Okta unveiled "Okta for AI Agents," a platform designed to register, monitor, and control AI agents using identity access management systems, aiming to improve security and management within organizations. ([techradar.com](https://www.techradar.com/pro/security/okta-unveils-new-framework-to-secure-and-protect-enterprise-ai-agents?utm_source=openai)) Nvidia formed the Nemotron Coalition, uniting eight AI companies to co-develop open frontier models on NVIDIA DGX Cloud, supporting the development of Nvidia’s upcoming Nemotron 4 model family. ([tomshardware.com](https://www.tomshardware.com/tech-industry/artificial-intelligence/nvidias-nemoclaw-coalition-brings-eight-ai-labs-together-to-build-open-frontier-models?utm_source=openai)) AWS introduced Amazon Bedrock AgentCore, a platform designed to simplify the development and deployment of advanced AI agents, emphasizing flexibility and scalability. ([techradar.com](https://www.techradar.com/pro/aws-looks-to-super-charge-ai-agents-with-amazon-bedrock-agentcore?utm_source=openai))


## Highlights:
- [Okta unveils new framework to secure and protect enterprise AI agents](https://www.techradar.com/pro/security/okta-unveils-new-framework-to-secure-and-protect-enterprise-ai-agents?utm_source=openai), Published on Tuesday, March 17
- [Nvidia's Nemotron coalition brings eight AI labs together to build open frontier models](https://www.tomshardware.com/tech-industry/artificial-intelligence/nvidias-nemoclaw-coalition-brings-eight-ai-labs-together-to-build-open-frontier-models?utm_source=openai), Published on Monday, March 16
- [AWS looks to super-charge AI agents with Amazon Bedrock AgentCore](https://www.techradar.com/pro/aws-looks-to-super-charge-ai-agents-with-amazon-bedrock-agentcore?utm_source=openai), Published on Wednesday, July 16 

### Have a look at the trace   
https://platform.openai.com/traces

### Now explore the structured output and include a description of the fields.

In [6]:
how_many_searches = 3
instructions = f"You are a helpful research assistant. Given a query, come up with a set of we searches \
to perform to best answer the query. Output {how_many_searches} terms to query for."

# use Pydantic to defnie the Schema of our response - this is known as "Structured Outputs"

class WebSearchItem(BaseModel):
    reason: str = Field(description = "Your reasoning for why this search is important to the query.")
    query: str = Field(description = "The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description = "The search web searches to perform to best answer the query.")

planner_agent = Agent(
    name = "Planner Agent",
    instructions = instructions,
    model = 'gpt-4o-mini',
    output_type = WebSearchPlan,

)

In [7]:
message = "Latest AI Agent frameworks in 2025"

with trace("Search"):
    result = await Runner.run(planner_agent, message)
    print(result.final_output)

searches=[WebSearchItem(reason='To discover the newest frameworks available in 2025 for developing AI agents, including updates on established frameworks.', query='latest AI agent frameworks 2025'), WebSearchItem(reason='To identify industry trends and leading AI agent frameworks that have emerged or evolved in 2025.', query='top AI agent frameworks 2025'), WebSearchItem(reason='To find specific features, comparisons, and documentation of the latest AI agent frameworks released in 2025.', query='AI agent frameworks comparison 2025')]


In [11]:
@function_tool
def send_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("coast092@gmail.com") # Change this to your verified email
    to_email = To("coast092@gmail.com") # Change this to your email
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return "success"

In [12]:
send_email

FunctionTool(name='send_email', description='Send out an email with the given subject and HTML body', params_json_schema={'properties': {'subject': {'title': 'Subject', 'type': 'string'}, 'html_body': {'title': 'Html Body', 'type': 'string'}}, 'required': ['subject', 'html_body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x0000026A88148E00>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None)

In [13]:
instructions = """You are able to send a nicely formatted HTML email based on a detailed report.
You will be provided  with a detailed report. You should use your tool to send one email, providing the report
converted into clean, well presented HTML with an appropriate subject line."""

email_agent = Agent(
    name = "Email Agent",
    instructions  = instructions,
    tools = [send_email],
    model = 'gpt-4o-mini',
)

In [16]:
instructions = (
    """You are a senior researcher tasked with writing a cohesive report for a research query. 
    You will be provided with the original query, and some initial research done by a research assistant 
    You should first come up with an outline for the report that describes the structrue and
    flow of the report. Then, generate the report and return that as your final output. 
    The final output should be in markdown format, and it should be lengthy and detailed.
    Aim for 5-10 paragraphs of the content, at most 1000 words.
    """
)

class ReportData(BaseModel):
    short_summary: str = Field(description = "A short 2-3 sentence summary of the findings.")
    markdown_report : str = Field(description = "The full report")
    follow_up_question : list[str] = Field(description = "Suggested topics to research further.")

writer_agent = Agent(
    name = "Writer Agent",
    instructions = instructions, 
    model = 'gpt-4o-mini',
    output_type = ReportData,
)


### The next 3 functions will plan and execute the search, using planner_agent and search_agent

In [20]:
async def plan_searches(query: str):
    """Use the planner_agent to plan which searches to run for the query"""
    print("Planning searches....")
    result = await Runner.run(planner_agent, f"Query: {query}")
    print(f"Will perform {len(result.final_output.searches)} searches")
    return result.final_output

async def perform_searches(search_plan: WebSearchPlan):
    """Call search() for each item in the search plan """
    print("Searching...")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results


async def search(item: WebSearchItem):
    """ Use the search agent to run a web search for each item in the search plan"""
    input = f"Search term : {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input)
    return result.final_output

### The next 2 functions write a report and email it

In [ ]:
async def write_report(query: str, search_results: list[str]):
    """Use the writer agent to write a report absed on the search results"""
    print("Thinking about report...")
    input = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input)
    print("Finished writing report")
    return result.final_output


async def send_email(report: ReportData):
    """Use the email agent to send an email with the report"""
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent!")
    return report

### Show time!

In [22]:
query = "Latest AI Agent frameworks in 2025"

with trace("Research trace"):
    print("Starting research...")
    search_plan = await plan_searches(query)
    search_results = await perform_searches(search_plan)
    report = await write_report(query, search_results)
    await send_email(report)
    print("Hooray!")

Starting research...
Planning searches....
Will perform 3 searches
Searching...
Finished searching
Thinking about report...
Finished writing report
Writing email...
Email send!
Hooray!


## As Always, take a look at the trace
https://platform.openai.com/traces